# CLPsych 2026 — Test Submission (adaptive-focus architecture)

In [ ]:
# 1. Setup
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader  # optional: confirms a GPU is available
%pip install -q -r requirements.txt

In [ ]:
# 2. Config
MODEL = 'Qwen/Qwen3.5-27B'
TEMPERATURE = 0.1
MAX_TOKENS = 100
RAG_K = 8
RAG_POOL = 20
RAG_ALPHA = 0.6

In [ ]:
# 3. Load data
from data_loader import load_all_timelines
from rag_index import RAGIndex

# Train data — used for RAG retrieval
train_posts = load_all_timelines('train_tasks12')
train_evidenced = [p for p in train_posts if p.has_evidence]
print(f'Train evidenced: {len(train_evidenced)}')

# Augmented data — for per-element A RAG
aug_posts = load_all_timelines('augmented_data')
aug_evidenced = [p for p in aug_posts if p.has_evidence]
print(f'Augmented: {len(aug_evidenced)}')

# Test data — target for predictions
test_posts = load_all_timelines('test_tasks12nolabels')
print(f'Test posts: {len(test_posts)}')

# RAG indexes (from train only)
rag_index_real = RAGIndex(train_evidenced)
rag_index_aug = RAGIndex(train_evidenced + aug_evidenced)
print(f'RAG real: {len(rag_index_real.posts)}, augmented: {len(rag_index_aug.posts)}')

In [ ]:
# 4. Build prompts (call 1: unified, call 3: per-element A)
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from tqdm import tqdm

target = test_posts

# Call 1: unified prompts
all_messages = []
rag_cache = []
for p in tqdm(target, desc='Unified'):
    rag = rag_index_real.retrieve(p.text, k=RAG_K, exclude_timeline=p.timeline_id,
                                  candidate_pool=RAG_POOL, diversity_alpha=RAG_ALPHA)
    all_messages.append(build_messages(p.text, rag))
    rag_cache.append(rag)

# Call 3: per-element A prompts
messages_A = []
for p in tqdm(target, desc='Element A'):
    rag_A = rag_index_aug.retrieve_for_element(p.text, 'A', k=RAG_K, exclude_timeline=p.timeline_id)
    messages_A.append(build_messages_per_element(p.text, 'A', rag_A))

print(f'Prompts: {len(all_messages)} unified + {len(messages_A)} element A')

In [ ]:
# 5. Load model
from vllm import LLM, SamplingParams

llm = LLM(MODEL, dtype='float16', max_model_len=8192)
params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
print('Model ready!')

In [ ]:
# 6. Inference: call 1 (unified) → call 2 (adaptive) → call 3 (per-element A)
from output_parser import parse_llm_output

# Call 1: unified
print('=== Call 1: unified ===')
print('Example prompt:')
for m in all_messages[0]:
    print(f"[{m['role'].upper()}]")
    print(m['content'])
    print()

outputs = llm.chat(all_messages, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Output: {outputs[0].outputs[0].text}')

# Parse unified to get maladaptive context for call 2
results_unified = [parse_llm_output(o.outputs[0].text.strip()) for o in outputs]

# Build adaptive-focused prompts with maladaptive context
messages_adaptive = []
for i, (p, u_pred) in enumerate(zip(target, results_unified)):
    messages_adaptive.append(build_messages_adaptive(p.text, rag_cache[i], u_pred['maladaptive']))

# Call 2: adaptive-focused
print('\n=== Call 2: adaptive-focused ===')
print('Example prompt:')
for m in messages_adaptive[0]:
    print(f"[{m['role'].upper()}]")
    print(m['content'])
    print()

outputs_adaptive = llm.chat(messages_adaptive, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Output: {outputs_adaptive[0].outputs[0].text}')

# Call 3: per-element A
print('\n=== Call 3: per-element A ===')
print('Example prompt:')
for m in messages_A[0]:
    print(f"[{m['role'].upper()}]")
    print(m['content'])
    print()

outputs_A = llm.chat(messages_A, sampling_params=params, chat_template_kwargs={'enable_thinking': False})
print(f'Output: {outputs_A[0].outputs[0].text}')

In [ ]:
# 7. Merge results
# Maladaptive: from unified (call 1), A from per-element A (call 3)
# Adaptive: elements from adaptive-focused (call 2), presence from unified (call 1)
# A (both valences): from per-element A (call 3)
import json
from pathlib import Path
from submission_formatter import save_submission

results = []
for i, post in enumerate(target):
    u_pred = results_unified[i]
    a_pred = parse_llm_output(outputs_adaptive[i].outputs[0].text.strip())
    a_elem_pred = parse_llm_output(outputs_A[i].outputs[0].text.strip())

    merged = {
        'maladaptive': {
            'elements': dict(u_pred['maladaptive']['elements']),
            'presence': u_pred['maladaptive']['presence'],
        },
        'adaptive': {
            'elements': dict(a_pred['adaptive']['elements']),
            'presence': u_pred['adaptive']['presence'],
        },
    }
    # Override A from per-element A (call 3) for both valences
    for v in ('adaptive', 'maladaptive'):
        merged[v]['elements'].pop('A', None)
        if 'A' in a_elem_pred[v]['elements']:
            merged[v]['elements']['A'] = a_elem_pred[v]['elements']['A']

    # Consistency: no elements → presence = 1
    for v in ('adaptive', 'maladaptive'):
        if not merged[v]['elements']:
            merged[v]['presence'] = 1

    results.append({
        'timeline_id': post.timeline_id,
        'post_id': post.post_id,
        'raw_output': outputs[i].outputs[0].text.strip(),
        'prediction': merged,
    })

print(f'{len(results)} predictions')

# Save raw results + submission
Path('outputs').mkdir(exist_ok=True)
with open('outputs/raw_results_test.json', 'w') as f:
    json.dump(results, f, indent=2)
save_submission(results, 'outputs/task1_pred.json')

In [ ]:
# 8. Validate submission
import sys, subprocess
if 'official_eval' not in sys.path:
    sys.path.insert(0, 'official_eval')

print('=== Submission Validation ===')
result = subprocess.run(
    [sys.executable, 'official_eval/validate_submission.py',
     '--task1', 'outputs/task1_pred.json', '--test-dir', 'test_tasks12nolabels'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    print('WARNING: validation failed!')

In [ ]:
# 9. Create submission zip
import zipfile
with zipfile.ZipFile('outputs/predictions.zip', 'w') as zf:
    zf.write('outputs/task1_pred.json', 'task1_pred.json')

print('Created predictions.zip')
